# How do you see that a deployed model gets worse, what do you do, and can you prove later what happened?

Question: one simulated classifier runs for 60 days. Labels arrive 7 days late. Which signals show that the model is slow, drifting, or broken? Which action fixes each fault? Can you trace every prediction back to its model, its data, and the people who changed it?

Setup: CPU only. No API key. The service is simulated, so the truth of every fault is known. The code asserts that truth before it uses it. Real libraries do the tool work: OpenTelemetry SDK (in-memory exporter), prometheus_client, MLflow (SQLite store, registry aliases), scikit-learn, SciPy. Part F and part G take their NVIDIA facts from files in `hub/sources/` of the exam repository. The file name is shown where it is used.

Objectives: sizing.1, sizing.2, sizing.4, sizing.5 (Run:AI), sizing.6, data.2, enterprise.2.

Terms used below:

- **Observability**: you can tell what a system does inside from the signals it sends out. The three signals are logs, metrics, and traces (`observability.txt`, `telemetry-observability.txt`).
- **Telemetry**: the stream of those signals, collected from applications and infrastructure.
- **Trace and span**: a trace is the path of one request. A span is one timed step in that path. A child span points to its parent.
- **SLA**: the latency or error limit you promise. An **SLA breach** is a measured value above that limit.
- **Drift**: the world moves away from the training data. **Covariate drift**: inputs change. **Concept drift**: the label rule changes, inputs stay.
- **PSI** (population stability index) and **KS** (Kolmogorov-Smirnov test): two ways to compare an input column against the training column.
- **MTTD / MTTR**: mean time to detect, mean time to restore. Both count from the minute the fault starts.
- **Champion / challenger**: the model version that serves, and the version that wants to replace it.
- **Shadow, canary, rollback**: shadow scores live traffic but serves nothing. Canary serves a small share. Rollback returns traffic to the champion.
- **Lineage and provenance**: lineage is the chain from a result back to its inputs. Provenance is the proof of where each input came from.
- **Runbook / playbook**: a runbook is executable steps for one incident. A playbook is the guide that picks the runbook, the owner, and the escalation.
- **Run:AI** quota, over-quota, preemption, GPU fraction: terms for part F, defined there.

## Setup

First cell installs the three libraries before any import. The next cells define helpers and the simulated service. Every part runs inside `part(name, fn)`: an error prints its traceback and the notebook goes on. Every table is printed with `print(df.to_string())` and saved as CSV in the output folder. The last cell shows the status of all parts.

In [ ]:
# 1. Install before the first import. Kaggle has uv. Nothing below is imported yet, so the kernel never holds an old version.
import shutil, subprocess, sys
PKGS = {"opentelemetry.sdk.trace": "opentelemetry-sdk==1.45.2", "prometheus_client": "prometheus-client", "mlflow": "mlflow==3.18.0"}
need = []
for mod, pkg in PKGS.items():
    try: __import__(mod)
    except Exception: need.append(pkg)
if need:
    cmd = ["uv", "pip", "install", "-q", "--system", "--break-system-packages"] if shutil.which("uv") else [sys.executable, "-m", "pip", "install", "-q"]
    r = subprocess.run(cmd + need, capture_output=True, text=True, timeout=900)
    print("install", need, "exit code", r.returncode); print(r.stdout[-400:], r.stderr[-600:])
import importlib.metadata as imd
for p in ["opentelemetry-sdk", "prometheus-client", "mlflow", "numpy", "pandas", "scikit-learn", "scipy"]:
    try: print(p, imd.version(p))
    except Exception: print(p, "not found")

In [ ]:
# 2. Setup: imports, the part() runner, and the small statistics helpers (Wilson interval, exact McNemar).
import hashlib, json, math, os, pickle, signal, subprocess, sys, threading, time, traceback
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression

SEED = 11
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
T0, STATUS, SERVERS = time.time(), [], []

def wilson(k, n, z=1.96):
    if n == 0: return 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

def ci(k, n):                                   # "rate [low,high]" in percent
    lo, hi = wilson(k, n)
    return f"{100 * k / max(n, 1):.1f} [{100 * lo:.1f},{100 * hi:.1f}]"

def mcnemar(a_ok, b_ok):                        # exact two-sided test on the same items: (a right b wrong, a wrong b right, p)
    a, b = np.asarray(a_ok, bool), np.asarray(b_ok, bool)
    n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
    return n10, n01, (1.0 if n10 + n01 == 0 else float(stats.binomtest(n01, n10 + n01, 0.5).pvalue))

def show(df, name):                             # print the table to the log and save it as CSV
    df.to_csv(f"{OUT}/{name}.csv", index=False); print(df.to_string(index=False)); print()

class Budget(Exception): pass

def part(name, fn, budget=300):
    """Run one part. Print the traceback on failure, record the status, and go on. SIGALRM enforces the time budget."""
    print(f"\n===== {name} ====="); t, st, err = time.time(), "ok", ""
    try:
        signal.signal(signal.SIGALRM, lambda *_: (_ for _ in ()).throw(Budget(f"time budget {budget} s exceeded"))); signal.alarm(budget)
    except ValueError: pass                     # not the main thread: no hard budget
    try: fn()
    except Exception as e:
        st, err = "FAILED", f"{type(e).__name__}: {e}"[:140]; traceback.print_exc(limit=8, file=sys.stdout)
    finally:
        try: signal.alarm(0)
        except ValueError: pass
    STATUS.append(dict(part=name, status=st, seconds=round(time.time() - t, 1), error=err)); print(f"[{name}] {st} in {time.time() - t:.1f} s")

# self-checks of the helpers
assert abs(wilson(50, 100)[0] - 0.4038) < 1e-3 and wilson(0, 0) == (0.0, 1.0)
assert mcnemar([1, 1, 1, 0], [1, 1, 1, 0])[2] == 1.0 and mcnemar([1] * 10, [0] * 10)[2] < 0.01
print("helpers ok, output folder:", OUT)

In [ ]:
# 3. The simulated service: a tabular classifier. Day d has 300 rows. The label of day d reaches you on day d+7.
NF, NDAY, DAILY, LAG = 5, 60, 300, 7
W = np.array([1.5, 1.0, 0.5, 0.0, 0.0])                 # x3 and x4 carry no signal: drift there should not hurt
ONSET = {"none": None, "gradual": 20, "sudden": 30, "concept": 30, "irrelevant": 30}

def batch(rng, d, scen, n=DAILY):
    X, w = rng.normal(size=(n, NF)), W.copy()
    if scen == "gradual" and d >= 20: X[:, 0] += 0.06 * (d - 19)     # covariate drift, grows each day
    if scen == "sudden" and d >= 30: X[:, 0] += 2.0                  # covariate drift, one step
    if scen == "irrelevant" and d >= 30: X[:, 4] += 3.0              # covariate drift in a column with weight 0
    if scen == "concept" and d >= 30: w[1] = -1.0                    # the label rule changes, X does not
    s = X @ w + 1.2 * X[:, 0] * X[:, 1] - (X[:, 0] ** 2 - 1) + rng.normal(0, 0.5, n)
    return X, (s > 0).astype(int)

def make_stream(seed, scen, days=NDAY):
    rng = np.random.default_rng(1000 * seed + 7)
    return [batch(rng, d, scen) for d in range(days)]

def make_ref(seed):                                     # day -1: training rows, a holdout, and the version 1 model
    rng = np.random.default_rng(1000 * seed + 3)
    X, y = batch(rng, -1, "none", 3000); Xh, yh = batch(rng, -1, "none", 3000)
    m = LogisticRegression(max_iter=500).fit(X, y)
    return dict(model=m, X=X, y=y, acc=float((m.predict(Xh) == yh).mean()), share=float(m.predict(X).mean()))

def psi(ref, x, bins=10):                               # population stability index, bins from reference quantiles
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1)[1:-1])
    e = np.clip(np.bincount(np.searchsorted(edges, ref), minlength=bins) / len(ref), 1e-4, None)
    a = np.clip(np.bincount(np.searchsorted(edges, x), minlength=bins) / len(x), 1e-4, None)
    return float(((a - e) * np.log(a / e)).sum())

def psi_max(ref, X): return max(psi(ref[:, j], X[:, j]) for j in range(NF))

# assert the truth of the simulator before any part uses it
r0 = np.random.default_rng(0).normal(size=5000)
assert psi(r0, r0) < 1e-9 and psi(r0, r0 + 3) > 1 and psi(r0, np.random.default_rng(1).normal(size=300)) < 0.1
REF0 = make_ref(0); rng0 = np.random.default_rng(5)
def acc_on(scen, d): X, y = batch(rng0, d, scen, 4000); return float((REF0["model"].predict(X) == y).mean())
base = acc_on("none", 10)
assert REF0["acc"] > 0.75 and abs(base - REF0["acc"]) < 0.03
assert acc_on("concept", 40) < base - 0.10 and acc_on("sudden", 40) < base - 0.10 and acc_on("gradual", 55) < base - 0.10
assert abs(acc_on("irrelevant", 40) - base) < 0.03 and abs(acc_on("none", 40) - base) < 0.03
print(f"truth ok: reference accuracy {REF0['acc']:.3f}; drift hurts for gradual, sudden and concept, not for irrelevant")

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Part | What you measure | Your guess | 
|---|---|---|
| A | A slow retrieval span and a failed tool span: can you find both from the trace alone? Which latency detector has fewer false alarms: one threshold on each request, or an EWMA? | |
| B | Which of PSI, KS, prediction share, and delayed-label accuracy sees concept drift? Which one raises an alarm for drift that does not hurt? | |
| C | Which retraining trigger gets the best accuracy? Which one retrains most often, and how many of those retrains make things worse? | |
| C | Does shadow alone stop a bad version? What does the canary stop? | |
| D | Can you rebuild the data hash of a model from the registry alone? What happens when you change one row? | |
| E | How much faster is an automated runbook than a manual one, for detection and for restore? | |
| F | Static quota against fair share against GPU fractions: which gives the lowest wait? Which one breaks the guaranteed quota? | |
| F | How does the cost of 1,000 requests change with batch size, and what does low utilization do to it? | |
| G | Given a few constraints, which deployment environment does the picker return? | |

## Part A. Observability

Question: when one request is slow or fails, can you see which step did it? The exam asks for dashboards, prompt/agent/retrieval tracing, logging and alerting, telemetry, and anomaly detection.

One request makes four spans (prompt, retrieval, model call, tool) under one root span. The code plants slow spans and failed tool calls. It finds them from the exported spans alone. Metrics go to Prometheus. Logs are JSON lines with the trace ID. Two latency detectors run: a threshold on each request, and an EWMA (exponentially weighted moving average). The setting is chosen on requests 0 to 599. It is measured on requests 600 to 1199.

In [ ]:
# 4. Part A: traces (OpenTelemetry), metrics (Prometheus), logs (JSON with trace ID), dashboard, anomaly detection.
INC = [(150, 190, "spike"), (380, 440, "ramp"), (520, 560, "spike"), (700, 750, "spike"), (880, 960, "ramp"), (1050, 1090, "spike")]
N_REQ, GAP_S, HALF = 1200, 2.0, 600
RA = {}

def part_a():
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor
    from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
    from opentelemetry.trace import Status, StatusCode, set_span_in_context
    from prometheus_client import CollectorRegistry, Counter, Histogram, generate_latest
    rng = np.random.default_rng(SEED)
    exp = InMemorySpanExporter(); prov = TracerProvider(); prov.add_span_processor(SimpleSpanProcessor(exp)); tr = prov.get_tracer("support-bot")
    reg = CollectorRegistry()
    REQ = Counter("requests_total", "requests", ["status"], registry=reg)
    LAT = Histogram("request_latency_seconds", "request latency", buckets=(0.2, 0.3, 0.4, 0.6, 1, 2), registry=reg)
    BASE = dict(prompt=0.005, retrieval=0.04, model_call=0.22, tool=0.06)
    def kind(i): return next(((k, s, e) for s, e, k in INC if s <= i < e), None)
    logs, planted, lat, failed, t_ns0 = [], {}, np.zeros(N_REQ), np.zeros(N_REQ, bool), 1_700_000_000 * 10 ** 9
    for i in range(N_REQ):
        d = {n: v * rng.lognormal(0, 0.12) for n, v in BASE.items()}; k = kind(i); slow = None
        if k and k[0] == "spike": d["retrieval"] *= 6; slow = "retrieval"
        elif k: d["model_call"] *= 1 + 0.6 * (i - k[1]) / (k[2] - k[1])
        if rng.random() < 0.03: d["retrieval"] *= 4; slow = "retrieval"          # cache miss: one slow request
        fail = bool(rng.random() < 0.02); cur = t_ns0 + int(i * GAP_S * 1e9)
        root = tr.start_span("request", start_time=cur); ctx = set_span_in_context(root)
        for name in ["prompt", "retrieval", "model_call", "tool"]:
            s = tr.start_span(name, context=ctx, start_time=cur); s.set_attribute("component", name)
            if name == "tool" and fail: s.set_status(Status(StatusCode.ERROR, "tool timeout"))
            cur += int(d[name] * 1e9); s.end(end_time=cur)
        if fail: root.set_status(Status(StatusCode.ERROR, "tool timeout"))
        root.end(end_time=cur); tid = format(root.get_span_context().trace_id, "032x")
        lat[i], failed[i] = (cur - t_ns0 - int(i * GAP_S * 1e9)) / 1e9, fail
        planted[tid] = dict(slow=slow, fail=fail)
        REQ.labels("error" if fail else "ok").inc(); LAT.observe(lat[i])
        logs.append(json.dumps(dict(ts=round(i * GAP_S, 1), level="ERROR" if fail else "INFO", msg="request done", trace_id=tid, latency_s=round(lat[i], 3))))
    spans = exp.get_finished_spans(); assert len(spans) == 5 * N_REQ
    by = {}
    for s in spans: by.setdefault(format(s.get_span_context().trace_id, "032x"), []).append(s)
    dur = lambda s: (s.end_time - s.start_time) / 1e9
    # A1. trace tree of the slowest request, read from the spans alone
    tid = max(by, key=lambda t: dur(next(s for s in by[t] if s.parent is None)))
    print("A1. trace tree of the slowest request, trace_id", tid[:12], "...")
    for s in sorted(by[tid], key=lambda s: (s.parent is not None, s.start_time)):
        print(("  " if s.parent else "") + f"{s.name:11s} {dur(s) * 1000:7.1f} ms  {'ERROR' if s.status.status_code.name == 'ERROR' else ''}")
    med = {n: float(np.median([dur(s) for s in spans if s.name == n])) for n in BASE}
    slow_found = [s.name for s in by[tid] if s.parent and dur(s) > 3 * med[s.name]]
    assert slow_found == [planted[tid]["slow"]], (slow_found, planted[tid])      # the trace alone names the planted slow span
    print(f"slow span found from the trace alone: {slow_found} (planted: {planted[tid]['slow']}), median span times ms {({n: round(1000 * v, 1) for n, v in med.items()})}")
    # A2. failed spans, from the status of the spans alone
    bad = {t: [s.name for s in v if s.status.status_code.name == "ERROR" and s.parent] for t, v in by.items()}
    found = {t for t, v in bad.items() if v}; truth = {t for t, p in planted.items() if p["fail"]}
    assert found == truth and all(v == ["tool"] for t, v in bad.items() if v)
    print(f"A2. failed requests found from span status: {len(found)} of {len(truth)} planted, failed span is always {sorted({x for v in bad.values() for x in v})}")
    # A3. metrics (Prometheus text format) and logs joined to traces by trace ID
    txt = generate_latest(reg).decode(); open(f"{OUT}/prometheus_metrics.txt", "w").write(txt)
    print("A3. metrics:", *[l for l in txt.splitlines() if l.startswith(("requests_total{", "request_latency_seconds_count", "request_latency_seconds_sum", 'request_latency_seconds_bucket{le="0.4"'))], sep="\n  ")
    open(f"{OUT}/request_logs.jsonl", "w").write("\n".join(logs))
    lg = {json.loads(l)["trace_id"]: json.loads(l) for l in logs}
    assert set(lg) == set(by) and len(lg) == N_REQ
    bt = next(iter(truth)); print(f"log joined to failed trace {bt[:8]}:", {k: v for k, v in lg[bt].items() if k != "trace_id"}, "| its spans:", sorted(s.name for s in by[bt]))
    # A5. anomaly detection on request latency: threshold on each request, and an EWMA, chosen on half 1, measured on half 2
    mu, sd = lat[:100].mean(), lat[:100].std(); a_ew = 0.2
    ew = np.zeros(N_REQ); ew[0] = lat[0]
    for i in range(1, N_REQ): ew[i] = a_ew * lat[i] + (1 - a_ew) * ew[i - 1]
    series = {"threshold": (lat, mu, sd), "ewma": (ew, mu, sd * math.sqrt(a_ew / (2 - a_ew)))}
    def episodes(al):
        eps = []
        for j in np.flatnonzero(al):
            if eps and j - eps[-1][1] <= 5: eps[-1][1] = j
            else: eps.append([j, j])
        return eps
    def score(al, lo, hi, inc=INC):
        a = al.copy(); a[:lo] = False; a[hi:] = False; incs = [(s, e) for s, e, _ in inc if lo <= s < hi]; eps = episodes(a)
        tp = sum(any(s <= x[0] <= e + 10 for s, e in incs) for x in eps); hit = [s for s, e in incs if any(s <= x[0] <= e + 10 for x in eps)]
        dl = [min(x[0] for x in eps if s <= x[0] <= e + 10) - s for s, e in incs if s in hit]
        p, r = tp / max(len(eps), 1), len(hit) / len(incs)
        return dict(eps=len(eps), tp=tp, hit=len(hit), n=len(incs), prec=p, rec=r, f1=0 if p + r == 0 else 2 * p * r / (p + r), delay_s=float(np.median(dl)) * GAP_S if dl else float("nan"))
    z = np.zeros(40, bool); z[10:20] = True; g1, g2, g3 = score(z, 0, 40, [(10, 20, "x")]), score(np.zeros(40, bool), 0, 40, [(10, 20, "x")]), score(np.roll(z, 25), 0, 40, [(10, 20, "x")])
    assert (g1["prec"], g1["rec"]) == (1.0, 1.0) and g2["rec"] == 0 and g3["prec"] == 0 and g3["rec"] == 0       # the grader on a right and a wrong alarm series
    rows, kb = [], {}
    for name, (x, m0, s0) in series.items():
        best = kb[name] = max([2, 3, 4, 5, 6], key=lambda k: score(x > m0 + k * s0, 0, HALF)["f1"])
        s2 = score(x > m0 + best * s0, HALF, N_REQ)
        rows.append(dict(detector=name, k_sigma=best, episodes=s2["eps"], precision=ci(s2["tp"], s2["eps"]), recall=ci(s2["hit"], s2["n"]), f1=round(s2["f1"], 2), delay_s=s2["delay_s"]))
    inc_share = np.mean([kind(i) is not None for i in range(HALF, N_REQ)])
    print("A5. anomaly detection, setting chosen on requests 0-599, measured on 600-1199. Episode = alarms within 5 requests.")
    print(f"baseline: an alarm at random hits an incident with probability {inc_share:.2f} (share of requests inside incidents); recall counts {sum(1 for s, _, _ in INC if s >= HALF)} incidents.")
    show(pd.DataFrame(rows), "A_anomaly_detection")
    RA.update(rows=rows, lat=lat, ew=ew, failed=failed, spans=spans)
    # A4. dashboard, four panels, windows of 30 requests (one minute)
    w = 30; nw = N_REQ // w; L = lat.reshape(nw, w)
    fig, ax = plt.subplots(2, 2, figsize=(11, 6)); x = np.arange(nw)
    ax[0, 0].plot(x, np.percentile(L, 95, axis=1)); ax[0, 0].axhline(0.6, color="r", ls="--"); ax[0, 0].set_title("p95 latency per minute (s), SLA 0.6")
    ax[0, 1].plot(x, failed.reshape(nw, w).mean(axis=1) * 100); ax[0, 1].set_title("error rate per minute (%)")
    for n in BASE: ax[1, 0].plot(x, np.array([dur(s) for t in by for s in by[t] if s.name == n]).reshape(nw, w).mean(axis=1), label=n)   # by keeps request order
    ax[1, 0].legend(fontsize=7); ax[1, 0].set_title("mean span time per minute (s)")
    ax[1, 1].plot(lat, lw=0.4, label="latency"); ax[1, 1].plot(ew, label="EWMA"); ax[1, 1].axhline(mu + kb["ewma"] * sd * math.sqrt(a_ew / (2 - a_ew)), color="r", ls="--"); ax[1, 1].legend(fontsize=7); ax[1, 1].set_title("request latency, EWMA, alarm line")
    fig.tight_layout(); fig.savefig(f"{OUT}/A_dashboard.png", dpi=90); plt.close(fig); print("saved", f"{OUT}/A_dashboard.png")

part("A observability", part_a)

## Part B. Drift

Question: which detector sees which drift? Post-training drift is the gap between the live data and the training data. Five scenarios run for 60 days with 5 seeds each: no drift, gradual covariate, sudden covariate, concept drift (labels change, inputs do not), and drift in a column the model ignores. Each detector has a loose and a strict threshold. An alarm needs 2 days in a row over the threshold. Delay counts days from the drift start to the first alarm. A false alarm is an alarm on a healthy day.

In [ ]:
# 5. Part B: four drift detectors, two thresholds each, five scenarios, five seeds.
THR = {"PSI": (0.1, 0.25), "KS": (0.05, 0.001), "pred_share": (2.0, 4.0), "label_acc": (0.05, 0.10)}   # loose, strict
DETS = [(n, i) for n in THR for i in (0, 1)]

def scores(stream, ref):
    m, p0, sc = ref["model"], ref["share"], {k: np.full(NDAY, np.nan) for k in THR}
    for d, (X, y) in enumerate(stream):
        sc["PSI"][d] = psi_max(ref["X"], X)
        sc["KS"][d] = min(1.0, NF * min(stats.ks_2samp(ref["X"][:, j], X[:, j]).pvalue for j in range(NF)))   # Bonferroni over columns
        sc["pred_share"][d] = abs(m.predict(X).mean() - p0) / math.sqrt(p0 * (1 - p0) / len(X))                # z of the positive-prediction share
        if d >= LAG: Xo, yo = stream[d - LAG]; sc["label_acc"][d] = ref["acc"] - (m.predict(Xo) == yo).mean() # accuracy drop, on labels that arrived
    return sc

def alarm(sc, name, i):                          # an alarm needs the threshold crossed on 2 days in a row. NaN (label not yet there) never alarms.
    a = (sc[name] < THR[name][i]) if name == "KS" else (sc[name] > THR[name][i])
    return np.r_[False, a[1:] & a[:-1]]

RB = {}
def part_b():
    t = {"KS": np.array([.5, .0001, .0001, .5]), "PSI": np.array([0, .3, .3, .3]), "pred_share": np.array([np.nan, 9, 9, 9]), "label_acc": np.zeros(4)}
    assert alarm(t, "KS", 1).tolist() == [False, False, True, False] and alarm(t, "PSI", 1).tolist() == [False, False, True, True] and not alarm(t, "pred_share", 0)[:2].any()   # alarm rule: 2 days in a row, NaN never
    seeds, res, rows_raw = range(5), {}, []
    for scen in ONSET:
        for s in seeds:
            ref, st = make_ref(s), make_stream(s, scen); sc = scores(st, ref)
            acc = np.array([(ref["model"].predict(X) == y).mean() for X, y in st]); res[scen, s] = (sc, acc)
            rows_raw += [dict(scenario=scen, seed=s, day=d, accuracy=acc[d], **{k: sc[k][d] for k in THR}) for d in range(NDAY)]
    pd.DataFrame(rows_raw).to_csv(f"{OUT}/B_daily_scores.csv", index=False)
    on = lambda sc_: ONSET[sc_] if ONSET[sc_] is not None else NDAY
    out, harm, cnt = [], {}, {}
    for scen in ONSET:
        a = np.array([res[scen, s][1] for s in seeds]); o = on(scen)
        harm[scen] = float(a[:, o:].mean() - a[:, :o].mean()) if o < NDAY else float(a[:, 30:].mean() - a[:, :30].mean())
        row = dict(scenario=scen, onset=ONSET[scen], acc_change=round(harm[scen], 3))
        for n, i in DETS:
            if ONSET[scen] is None:
                k = sum(int(alarm(res[scen, s][0], n, i).sum()) for s in seeds); row[f"{n}@{THR[n][i]}"] = f"fa {k}/{NDAY * len(seeds)}"; continue
            first = [next((d - o for d in range(o, NDAY) if alarm(res[scen, s][0], n, i)[d]), None) for s in seeds]; got = [x for x in first if x is not None]
            cnt[scen, n, i] = len(got); row[f"{n}@{THR[n][i]}"] = f"{int(np.median(got))}d {len(got)}/5" if got else "- 0/5"
        out.append(row)
    print("B1. detection delay in days from drift start (median over seeds) and seeds that alarmed. acc_change = mean accuracy after minus before the drift start.")
    (a5, b5), (a1, b1) = wilson(5, 5), wilson(1, 5); print(f"With 5 seeds, 5/5 has a Wilson 95% interval of {100 * a5:.0f}% to {100 * b5:.0f}%, and 1/5 has {100 * a1:.0f}% to {100 * b1:.0f}%.")
    show(pd.DataFrame(out), "B_detection_delay")
    fa = []
    for n, i in DETS:
        k = tot = 0
        for scen in ONSET:
            for s in seeds:
                sc = res[scen, s][0]; a = alarm(sc, n, i)[:on(scen)]; ok = ~np.isnan(sc[n][:on(scen)]); k += int(a[ok].sum()); tot += int(ok.sum())
        fa.append(dict(detector=f"{n}@{THR[n][i]}", false_alarm_days=k, healthy_days=tot, false_alarm_rate_pct=ci(k, tot)))
    print("B2. false alarms: alarms on healthy days (before the drift start, and all days of the no-drift scenario), pooled over seeds.")
    show(pd.DataFrame(fa), "B_false_alarms")
    print("B3. what each detector sees (loose threshold, alarmed in at least 4 of 5 seeds), computed from B1:")
    for r in out:
        if ONSET[r["scenario"]] is None: continue
        see = [n for n in THR if cnt[r["scenario"], n, 0] >= 4]
        print(f"  {r['scenario']:11s} accuracy change {r['acc_change']:+.3f} | alarmed: {see if see else 'none'} | silent: {[n for n in THR if n not in see]}")
    assert harm["concept"] < -0.1 and harm["sudden"] < -0.1 and abs(harm["irrelevant"]) < 0.03 and abs(harm["none"]) < 0.03   # truth of the simulator
    RB.update(res=res)

part("B drift", part_b)

## Part C. Lifecycle, retraining triggers

Question: when should the model retrain? Four triggers run on the same 60-day streams: never, a weekly schedule, a drift alert (PSI above 0.25), and a performance alert (delayed-label accuracy 8 points under the reference). Three streams have drift. The fourth, `label_fault`, has no drift. Its label feed is flipped on days 36 to 44. A retrain uses the last 14 days that have labels. It never sees the last 7 days. Retrains are at least 7 days apart. A retrain is bad when the new model is more than 2 points worse than the old model on the next 7 days. The simulator knows those labels. A real team does not.

In [ ]:
# 6. Part C (1): four retraining triggers on the same streams. Row-level correctness allows an exact McNemar test against "never".
RC = {}
def run_policy(policy, stream, ref, fault=False):
    lab = lambda j: 1 - stream[j][1] if fault and 36 <= j <= 44 else stream[j][1]      # label fault: the label feed is flipped on days 36..44. Evaluation uses the true labels.
    m, last, ok, retr, bad = ref["model"], -99, [], [], 0
    for d in range(NDAY):
        X, y = stream[d]; ok.append(m.predict(X) == y)
        due = ((policy == "weekly" and d % 7 == 0) or (policy == "drift" and psi_max(ref["X"], X) > 0.25)
               or (policy == "perf" and d >= LAG and ref["acc"] - (m.predict(stream[d - LAG][0]) == lab(d - LAG)).mean() > 0.08))
        if due and d >= 14 and d - last >= 7 and d < NDAY - 7:
            lo, hi = max(0, d - LAG - 13), d - LAG                           # last 14 days that have labels
            Xw, yw = np.vstack([stream[j][0] for j in range(lo, hi + 1)]), np.concatenate([lab(j) for j in range(lo, hi + 1)])
            new = LogisticRegression(max_iter=500).fit(Xw, yw); nxt = range(d + 1, d + 8)
            a_old = np.mean([(m.predict(stream[j][0]) == stream[j][1]).mean() for j in nxt]); a_new = np.mean([(new.predict(stream[j][0]) == stream[j][1]).mean() for j in nxt])
            bad += int(a_new < a_old - 0.02); retr.append(d); m, last = new, d      # the model changes from day d+1 on
    return np.concatenate(ok), len(retr), bad

def part_c1():
    rows, pol = [], ["never", "weekly", "drift", "perf"]
    worlds = {"gradual": ("gradual", False), "sudden": ("sudden", False), "concept": ("concept", False), "label_fault": ("none", True)}
    for scen, (sc_, fault) in worlds.items():
        got = {p: [run_policy(p, make_stream(s, sc_), make_ref(s), fault) for s in range(3)] for p in pol}
        base = np.concatenate([g[0] for g in got["never"]])
        for p in pol:
            ok = np.concatenate([g[0] for g in got[p]]); n10, n01, pv = mcnemar(ok, base)
            rows.append(dict(scenario=scen, trigger=p, accuracy=ci(int(ok.sum()), len(ok)), retrains=sum(g[1] for g in got[p]), bad_retrains=sum(g[2] for g in got[p]),
                             vs_never=("-" if p == "never" else f"{n10} better {n01} worse p={'<1e-100' if pv < 1e-100 else f'{pv:.1e}'}")))
    print("C1. accuracy over 60 days x 3 seeds (pooled rows), retrains summed over the 3 seeds, paired exact McNemar on the same rows against 'never'.")
    show(pd.DataFrame(rows), "C_retraining_triggers"); RC["rows"] = rows
    nv = next(r for r in rows if r["scenario"] == "label_fault" and r["trigger"] == "never")      # truth: evaluation uses the true labels, not the flipped feed
    assert float(nv["accuracy"].split()[0]) > 100 * make_ref(0)["acc"] - 5, nv
    best = {s: max((r for r in rows if r["scenario"] == s), key=lambda r: float(r["accuracy"].split()[0]))["trigger"] for s in worlds}
    print("best trigger by accuracy per scenario (computed from the table above):", best)

part("C1 retraining triggers", part_c1)

## Part C. Lifecycle, versions and release

Question: how does a new version reach users, and what stops a bad one? MLflow keeps the runs, the registry versions, and the `champion` and `challenger` aliases. A release has a shadow stage, a canary stage, a decision, and an automatic rollback. This part runs 120 days, so labels arrive before the canary gate. Version 2 is a good retrain after the concept drift. Version 3 has a planted train and serve skew bug. Two columns swap at training and not at serving. The offline test swaps them too, so it looks fine.

In [ ]:
# 7. Part C (2): MLflow runs and registry, aliases, shadow, canary, automatic rollback. 120 days so that labels arrive before the gate.
MODEL, DAYS, EVENTS, MODELS, VWIN = "tab-classifier", 120, [], {}, {1: (-1, -1), 2: (31, 38), 3: (66, 73)}
RD = {}

def row_hashes(df):                              # one SHA-256 per row, from the row values
    c = [f"x{j}" for j in range(NF)]
    return [hashlib.sha256(f"{r}|{u}|{'|'.join(f'{v:.6f}' for v in xs)}|{y}".encode()).hexdigest()
            for r, u, xs, y in zip(df.row_id, df.customer, df[c].to_numpy(), df.y)]

def data_hash(hs): return hashlib.sha256("\n".join(hs).encode()).hexdigest()      # rows are in row_id order

def part_c2():
    import mlflow, tempfile
    from mlflow import MlflowClient
    db = f"{OUT}/mlflow.db"
    if os.path.exists(db): os.remove(db)
    mlflow.set_tracking_uri(f"sqlite:///{db}"); mlflow.set_experiment(MODEL); cl = MlflowClient(); cl.create_registered_model(MODEL)
    rng = np.random.default_rng(42); ref = make_ref(0); stream = make_stream(0, "concept", DAYS)
    parts = [pd.DataFrame(ref["X"], columns=[f"x{j}" for j in range(NF)]).assign(day=-1, y=ref["y"], row_id=[f"ref-{i:04d}" for i in range(len(ref["y"]))])]
    for d, (X, y) in enumerate(stream): parts.append(pd.DataFrame(X, columns=[f"x{j}" for j in range(NF)]).assign(day=d, y=y, row_id=[f"d{d:03d}-{i:03d}" for i in range(len(y))]))
    store = pd.concat(parts, ignore_index=True); store["customer"] = rng.integers(0, 3000, len(store)); store = store.sort_values("row_id").reset_index(drop=True)
    store["h"] = row_hashes(store); RD["store"] = store
    cols = [f"x{j}" for j in range(NF)]; PERM = [0, 2, 1, 3, 4]          # the planted bug: columns 1 and 2 swap at training only

    def train(ver, bug=False):
        lo, hi = VWIN[ver]; w = store[(store.day >= lo) & (store.day <= hi)]; ho = store[store.day == (hi + 1 if ver > 1 else 99999)]
        c = PERM if bug else slice(None); m = LogisticRegression(max_iter=500).fit(w[cols].to_numpy()[:, c], w.y)
        if ver == 1: hold = None
        else: hold = float((m.predict(ho[cols].to_numpy()[:, c]) == ho.y).mean())            # offline test: same pipeline as training
        with mlflow.start_run(run_name=f"train-v{ver}") as run:
            rid = run.info.run_id; mlflow.log_params(dict(window=f"days {lo}..{hi}", n_rows=len(w), model="LogisticRegression", max_iter=500, seed=0))
            if hold is not None: mlflow.log_metric("offline_holdout_acc", hold)
            dh = data_hash(list(w.h)); mlflow.set_tag("data_hash", dh); tmp = tempfile.mkdtemp()
            w[["row_id", "h"]].rename(columns={"h": "row_hash"}).to_csv(f"{tmp}/rows.csv", index=False); pickle.dump(m, open(f"{tmp}/model.pkl", "wb"))
            mlflow.log_artifact(f"{tmp}/rows.csv"); mlflow.log_artifact(f"{tmp}/model.pkl")
        mv = cl.create_model_version(MODEL, f"runs:/{rid}/model.pkl", rid, tags={"data_hash": dh}); v = int(mv.version); assert v == ver
        MODELS[v] = m                               # what the serving host holds in memory (skew bug: raw columns reach the swapped model)
        return v, hold, rid

    def champion(): return int(cl.get_model_version_by_alias(MODEL, "champion").version)
    v1, _, rid1 = train(1); cl.set_registered_model_alias(MODEL, "champion", str(v1)); EVENTS.append(dict(day=0, actor="ml-platform", action="register+promote", version=1))
    back = pickle.load(open(mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{rid1}/model.pkl"), "rb"))     # the registry artifact round trip
    assert (back.predict(ref["X"]) == MODELS[1].predict(ref["X"])).all()
    champ_at, canary, stages = np.ones(DAYS, int), {}, []

    def cycle(ver, bug):
        lo, hi = VWIN[ver]; t0 = hi + LAG; v, hold, _ = train(ver, bug); cv = champion()
        cl.set_registered_model_alias(MODEL, "challenger", str(v)); EVENTS.append(dict(day=t0, actor="retrain-job", action="register", version=v, offline_holdout_acc=round(hold, 3)))
        sh = [stream[d][0] for d in range(t0, t0 + 3)]                                                    # shadow: days t0..t0+2, nothing served
        sc_, sh_ = np.mean([MODELS[cv].predict(X).mean() for X in sh]), np.mean([MODELS[v].predict(X).mean() for X in sh])
        g1 = abs(sc_ - sh_) <= 0.15
        stages.append(dict(version=v, stage="offline test", metric="holdout acc", value=round(hold, 3), decision="pass" if hold > 0.7 else "fail"))
        stages.append(dict(version=v, stage="shadow", metric="positive-share gap", value=round(abs(sc_ - sh_), 3), decision="pass" if g1 else "stop"))
        EVENTS.append(dict(day=t0 + 3, actor="release-gate", action="shadow", version=v, passed=bool(g1)))
        if not g1: cl.delete_registered_model_alias(MODEL, "challenger"); return
        okc, okh, served = [], [], 0                                                                       # canary: 20% of rows, days t0+3..t0+6
        for d in range(t0 + 3, t0 + 7):
            X, y = stream[d]; mask = rng.random(len(X)) < 0.2; pc, ph = MODELS[cv].predict(X), MODELS[v].predict(X)
            ver_arr, pred = np.full(len(X), cv), pc.copy(); ver_arr[mask], pred[mask] = v, ph[mask]; canary[d] = (ver_arr, pred)
            okc += list(pc[mask] == y[mask]); okh += list(ph[mask] == y[mask])
        g = t0 + 6 + LAG; n10, n01, p = mcnemar(okc, okh); ac, ah = np.mean(okc), np.mean(okh)           # gate day: labels of the last canary day have arrived
        dec = "promote" if ah > ac and p < 0.05 else ("rollback" if ah < ac and p < 0.05 else "hold")
        stages.append(dict(version=v, stage="canary", metric=f"acc v{v} vs v{cv} ({len(okc)} rows)", value=f"{ah:.3f} vs {ac:.3f} p={p:.2g}", decision=dec))
        if dec == "promote": cl.set_registered_model_alias(MODEL, "champion", str(v)); cl.set_model_version_tag(MODEL, str(v), "status", "promoted"); cl.set_model_version_tag(MODEL, str(cv), "status", "retired"); champ_at[g + 1:] = v
        else: cl.set_model_version_tag(MODEL, str(v), "status", "rolled_back" if dec == "rollback" else "held")
        cl.delete_registered_model_alias(MODEL, "challenger"); EVENTS.append(dict(day=g, actor="release-gate", action=dec, version=v, canary_acc=round(float(ah), 3), champion_acc=round(float(ac), 3)))
        # blast radius: wrong answers added in the 10 days after the shadow gate, with the canary against a full switch at the shadow gate
        days = range(t0 + 3, t0 + 13); full = sum(((MODELS[v].predict(stream[d][0]) != stream[d][1]).sum() - (MODELS[cv].predict(stream[d][0]) != stream[d][1]).sum()) for d in days)
        part = (len(okh) - sum(okh)) - (len(okc) - sum(okc))
        RD.setdefault("blast", []).append(dict(version=v, decision=dec, extra_wrong_with_canary=int(part), extra_wrong_if_switched_at_shadow_gate=int(full)))
    cycle(2, False); cycle(3, True)
    print("C2. release stages (offline test, shadow, canary). Gate day = last canary day + 7 label delay. Canary rows are scored by both versions, so McNemar is paired.")
    show(pd.DataFrame(stages), "C_release_stages")
    print("Blast radius, in wrong answers added (positive = worse than the champion):"); show(pd.DataFrame(RD["blast"]), "C_blast_radius")
    reg = [dict(version=int(m.version), run_id=m.run_id[:8], data_hash=m.tags["data_hash"][:10], status=m.tags.get("status", "-"), window=f"days {VWIN[int(m.version)][0]}..{VWIN[int(m.version)][1]}") for m in cl.search_model_versions(f"name = '{MODEL}'")]
    al = cl.get_registered_model(MODEL).aliases; print("registry aliases:", al); show(pd.DataFrame(reg).sort_values("version"), "C_registry")
    print("MLflow runs:"); r = mlflow.search_runs(); print(r[[c for c in r.columns if c in ("run_id", "tags.mlflow.runName") or c.startswith("metrics.")]].to_string(index=False))
    assert champion() == 2 and "challenger" not in al
    rows = []                                                   # served predictions: version and prediction per row
    for d in range(DAYS):
        X, y = stream[d]
        if d in canary: ver_arr, pred = canary[d]
        else: ver_arr = np.full(len(X), int(champ_at[d])); pred = MODELS[int(champ_at[d])].predict(X)
        rows.append(pd.DataFrame(dict(prediction_id=[f"p{d:03d}-{i:03d}" for i in range(len(y))], row_id=[f"d{d:03d}-{i:03d}" for i in range(len(y))], day=d, version=ver_arr, pred=pred, y=y)))
    pr = pd.concat(rows, ignore_index=True); pr["correct"] = pr.pred == pr.y; RD["pred"] = pr; pr.to_csv(f"{OUT}/C_predictions.csv", index=False)
    print("served accuracy by version:", {int(v): round(float(g.correct.mean()), 3) for v, g in pr.groupby("version")}, "| rows", len(pr), "| baseline, always the majority label:", round(float(max(pr.y.mean(), 1 - pr.y.mean())), 3))
    RD["cl"] = cl; RD["mlflow"] = mlflow

part("C2 versions and release", part_c2, budget=900)

## Part D. Governance

Question: can you prove later what happened? The exam asks for data governance standards, lineage and provenance for auditability, and model risk management. Here a standard is a written rule that code checks. The code walks from one prediction ID back to its model version, run, parameters, data hash, and source rows. It changes one row to test the hash. It answers a deletion request. It keeps an append-only audit log. Each entry holds the hash of the entry before it. It writes a model card and a risk tier. Sources: `lineage-tracking.txt` (lineage for auditing) and `mastering-llm-techniques-llmops.txt` (versioned models with model cards in a registry, model risk assessment).

In [ ]:
# 8. Part D: lineage, tamper test, deletion request, hash-chained audit log, model card, risk tier.
def chain(events):                               # append-only log: each entry holds the hash of the entry before
    log, prev = [], "0" * 64
    for i, e in enumerate(events):
        h = hashlib.sha256((prev + json.dumps(e, sort_keys=True)).encode()).hexdigest(); log.append(dict(seq=i, event=e, prev=prev, hash=h)); prev = h
    return log

def verify(log):                                 # index of the first bad entry, or None
    prev = "0" * 64
    for i, x in enumerate(log):
        if x["prev"] != prev or x["hash"] != hashlib.sha256((prev + json.dumps(x["event"], sort_keys=True)).encode()).hexdigest(): return i
        prev = x["hash"]

def risk_tier(impact, autonomy, sensitivity, reversibility):    # each factor 0..2. Weights are this notebook's own rule, not a standard.
    s = impact + autonomy + sensitivity + reversibility
    return ("low" if s <= 2 else "medium" if s <= 5 else "high"), s

TIER_CONTROLS = {"low": ["monitoring"], "medium": ["monitoring", "drift alerts", "audit log", "human review on appeal"],
                 "high": ["monitoring", "drift alerts", "audit log", "human approval before release", "independent validation"]}

def part_d():
    store, pred, cl, mlflow = RD["store"], RD["pred"], RD["cl"], RD["mlflow"]
    cols = [f"x{j}" for j in range(NF)]
    # D1. lineage: one prediction ID back to the source rows
    pid = pred[(pred.version == 2) & (pred.day == 70)].prediction_id.iloc[5]; p = pred.set_index("prediction_id").loc[pid]
    mv = cl.get_model_version(MODEL, str(int(p.version))); run = cl.get_run(mv.run_id)
    rows = pd.read_csv(mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{mv.run_id}/rows.csv")).sort_values("row_id")
    src = store[store.row_id.isin(rows.row_id)].sort_values("row_id"); h_now = row_hashes(src)
    print("D1. lineage of one prediction (walk back, one hop per line):")
    print(f"  prediction {pid}: model version {int(p.version)}, answered {int(p.pred)}, input row {p.row_id}")
    print(f"  model version {mv.name} v{mv.version}, registry source {mv.source}")
    print(f"  run {mv.run_id[:8]}, params {run.data.params}")
    print(f"  data hash in the registry {mv.tags['data_hash'][:16]}..., in the run {run.data.tags['data_hash'][:16]}..., recomputed from the store {data_hash(h_now)[:16]}...")
    print(f"  source rows: {len(rows)} rows, first {rows.row_id.iloc[0]}, last {rows.row_id.iloc[-1]}, days {sorted(src.day.unique())[0]}..{sorted(src.day.unique())[-1]}")
    assert mv.tags["data_hash"] == run.data.tags["data_hash"] == data_hash(h_now) and p.row_id in set(store.row_id)
    # D2. tamper test: change one source row, the hash must not match, and the row hash list names the row
    bad = store.copy(); victim = rows.row_id.iloc[len(rows) // 2]; bad.loc[bad.row_id == victim, "x2"] += 0.001
    src2 = bad[bad.row_id.isin(rows.row_id)].sort_values("row_id"); h2 = row_hashes(src2)
    found = list(rows.row_id[np.array(h2) != rows.row_hash.to_numpy()])
    print(f"D2. tamper test: one value in row {victim} changed by 0.001. Data hash stored {mv.tags['data_hash'][:12]}, now {data_hash(h2)[:12]}, match: {data_hash(h2) == mv.tags['data_hash']}. Rows that differ from the logged row hashes: {found}")
    assert data_hash(h2) != mv.tags["data_hash"] and found == [victim]
    # D3. deletion request: which versions trained on rows of one customer
    versions = [int(m.version) for m in cl.search_model_versions(f"name = '{MODEL}'")]
    def trained_on(cust):
        ids = store.row_id[store.customer == cust]; out = []
        for v in versions:
            rid = cl.get_model_version(MODEL, str(v)).run_id; used = set(pd.read_csv(mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{rid}/rows.csv")).row_id)
            if ids.isin(used).any(): out.append(v)
        return out
    truth = lambda cust: [v for v, (lo, hi) in VWIN.items() if ((store.customer == cust) & (store.day >= lo) & (store.day <= hi)).any()]
    cust = max(sorted(store.customer.unique()), key=lambda c: len(truth(c)))
    got = trained_on(cust); assert got == truth(cust) and len(got) >= 2
    for v in got: cl.set_model_version_tag(MODEL, str(v), "deletion_request", "DR-1001")
    EVENTS.append(dict(day=100, actor="privacy-office", action="deletion_request", customer=int(cust), versions=got))
    print(f"D3. deletion request for customer {cust}: {int((store.customer == cust).sum())} rows. Versions trained on them: {got} (checked against the training windows). Tagged DR-1001. Action: retrain without these rows.")
    # D4. audit log with a hash chain
    log = chain(EVENTS); anchor = log[-1]["hash"]; k = len(log) // 2; assert verify(log) is None
    f1 = json.loads(json.dumps(log)); f1[k]["event"]["actor"] = "intruder"
    f2 = json.loads(json.dumps(f1)); f2[k]["hash"] = hashlib.sha256((f2[k]["prev"] + json.dumps(f2[k]["event"], sort_keys=True)).encode()).hexdigest()    # forger also fixes the entry's own hash
    f3 = json.loads(json.dumps(log)); del f3[k]
    f4 = json.loads(json.dumps(log))[:-2]
    res = [("clean log", verify(log)), (f"edit entry {k}", verify(f1)), (f"edit entry {k} and fix its hash", verify(f2)), (f"delete entry {k}", verify(f3)), ("cut last 2 entries", verify(f4))]
    print(f"D4. audit log: {len(log)} entries, head hash (anchor) {anchor[:16]}...")
    for n, r in res: print(f"  {n:34s} -> first bad entry: {r}")
    print(f"  cut tail passes the chain, but the head hash differs from the anchor stored elsewhere: {f4[-1]['hash'] != anchor}")
    assert res[0][1] is None and res[1][1] == k and res[2][1] == k + 1 and res[3][1] == k and res[4][1] is None and f4[-1]["hash"] != anchor
    pd.DataFrame([dict(seq=x["seq"], hash=x["hash"], **{a: b for a, b in x["event"].items() if a != "versions"}) for x in log]).to_csv(f"{OUT}/D_audit_log.csv", index=False)
    # D5. model card and risk tier
    ps = pred[pred.version == 2]; tier, score = risk_tier(impact=1, autonomy=2, sensitivity=1, reversibility=0)
    assert risk_tier(0, 0, 1, 0)[0] == "low" and risk_tier(2, 2, 2, 1)[0] == "high" and tier == "medium"
    card = dict(model=MODEL, version=2, role="champion", intended_use="rank requests for the simulated support service", not_for="decisions without human appeal",
                training_data=dict(rows=len(rows), days="31..38", data_hash=mv.tags["data_hash"], run_id=mv.run_id),
                served_accuracy=ci(int(ps.correct.sum()), len(ps)), baseline_majority_label_pct=round(100 * float(max(ps.y.mean(), 1 - ps.y.mean())), 1), limits="linear model, no calibration check, one simulated concept drift seen",
                monitoring=["PSI", "KS", "prediction share", "delayed-label accuracy (7 day lag)"], retraining="alert on delayed-label accuracy, 14 day window",
                risk=dict(tier=tier, score=score, controls=TIER_CONTROLS[tier]), audit_head=anchor[:16], owner="ml-platform (simulated)")
    json.dump(card, open(f"{OUT}/D_model_card.json", "w"), indent=1); print("D5. model card:"); print(json.dumps(card, indent=1)[:1800])
    rules = [("every training row has an ID and a hash", store.h.notna().all() and store.row_id.is_unique), ("every model version records a data hash", all("data_hash" in cl.get_model_version(MODEL, str(v)).tags for v in versions)),
             ("every prediction names its model version", bool(pred.version.notna().all())), ("a deletion request can be answered", got == truth(cust)), ("the audit log verifies", verify(log) is None)]
    print("data governance standards checked by code:"); show(pd.DataFrame([dict(rule=a, holds=bool(b)) for a, b in rules]), "D_governance_checks"); assert all(b for _, b in rules)

part("D governance", part_d, budget=600)

## Part E. Runbooks and playbooks

Question: how much do automated runbooks save? Four faults enter a 300-minute simulated service: a latency breach, input drift, a bad release, and a stopped data feed. An alert rule needs 3 minutes in a row over the limit. A runbook is a list of steps. Each step has a machine time. A playbook maps each alert to a runbook, an owner role, and an escalation time. An unknown alert goes to a human.

Human time in manual mode is an assumption, not a measurement: a dashboard check every 15 minutes, 1 to 5 minutes to acknowledge, 5 to 12 minutes to diagnose, 2 to 6 minutes per step. Automated mode adds 1 minute from alert to action. Machine time per step is the same in both modes. The toolchain is a real local HTTP webhook with a ticket store: alert, ticket JSON, action, closed ticket. Source for the runbook form: `runbooks.txt` (NVCF control plane runbooks with commands, health checks, and rollback).

In [ ]:
# 9. Part E: runbooks, playbook, MTTD and MTTR with manual and automated steps, a real local webhook and ticket store.
import http.server, urllib.request
SLA_S, FOR_MIN, T_END = 0.60, 3, 300
INCS = {"latency": 60, "drift": 120, "bad_release": 180, "feed": 240}          # injected fault start minute
BREACH = {"latency": lambda m: m["p95"] > SLA_S, "drift": lambda m: m["psi"] > 0.25, "bad_release": lambda m: m["err"] > 0.05, "feed": lambda m: m["rows"] < 50}
RUNBOOK = {   # steps: (what the step does, machine minutes). The last step verifies the health check.
    "latency": [("check replica health", 0), ("restart the slow replica", 1), ("verify p95 under the SLA", 1)],
    "drift": [("confirm PSI above 0.25", 0), ("start a retrain on the last 14 labeled days", 8), ("canary and promote the new version", 3), ("verify PSI against the new reference", 1)],
    "bad_release": [("compare error rate of old and new version", 0), ("move the champion alias back to the previous version", 1), ("verify the error rate", 1)],
    "feed": [("check the ingestion process", 0), ("restart the ingestion job", 2), ("verify rows per minute", 1)]}
PLAYBOOK = {"latency": dict(owner="serving on-call", severity="P2", escalate_after_min=10), "drift": dict(owner="ml on-call", severity="P3", escalate_after_min=30),
            "bad_release": dict(owner="release manager", severity="P1", escalate_after_min=10), "feed": dict(owner="data on-call", severity="P2", escalate_after_min=10)}
def playbook(alert):                              # the decision guide: alert -> runbook, owner, escalation. Unknown alerts go to a human.
    return dict(runbook=alert, **PLAYBOOK[alert]) if alert in PLAYBOOK else dict(runbook=None, owner="on-call lead", severity="unknown", escalate_after_min=0)
TICKETS = {}
class Hook(http.server.BaseHTTPRequestHandler):   # a tiny alert receiver and ticket store (JSON over HTTP)
    def _send(self, obj, code=200):
        b = json.dumps(obj).encode(); self.send_response(code); self.send_header("Content-Type", "application/json"); self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
    def _body(self): return json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
    def do_POST(self):
        a = self._body()["alerts"][0]; tid = f"OPS-{len(TICKETS) + 1}"; pb = playbook(a["labels"]["alertname"])
        TICKETS[tid] = dict(id=tid, title=f"alert {a['labels']['alertname']}", status="open", severity=pb["severity"], owner=pb["owner"], opened_min=a["startsAt"], timeline=[])
        self._send(TICKETS[tid], 201)
    def do_PATCH(self):
        t = TICKETS[self.path.rsplit("/", 1)[1]]; b = self._body(); t["status"] = b.get("status", t["status"])
        if "comment" in b: t["timeline"].append(b["comment"])
        if t["status"] == "closed": t["closed_min"] = b["minute"]
        self._send(t)
    def log_message(self, *a): pass
def call(base, method, path, obj):
    r = urllib.request.Request(base + path, data=json.dumps(obj).encode(), method=method, headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(r, timeout=5))

def simulate(mode, seed, hook=None):
    rng = np.random.default_rng(seed); S = dict.fromkeys(INCS, False); streak = dict.fromkeys(INCS, 0); busy, pend, det, res, fa, esc = set(), [], {}, {}, 0, 0
    phase, tk = int(rng.integers(0, 15)), {}
    for t in range(T_END):
        for k, v in INCS.items():
            if t == v: S[k] = True
        for fix_t, k in [x for x in pend if x[0] <= t]:
            S[k] = False; pend.remove((fix_t, k)); res[k] = t; busy.discard(k)
            if hook: call(hook, "PATCH", f"/tickets/{tk[k]}", dict(status="closed", minute=t, comment=f"min {t}: health check passed, ticket closed"))
        m = dict(p95=0.30 * rng.lognormal(0, 0.05) * (3 if S["latency"] else 1), psi=0.04 + rng.normal(0, 0.01) + (0.4 if S["drift"] else 0),
                 err=max(0, 0.01 + rng.normal(0, 0.004)) + (0.25 if S["bad_release"] else 0), rows=0 if S["feed"] else int(rng.normal(300, 10)))
        for k in INCS:
            streak[k] = streak[k] + 1 if BREACH[k](m) else 0
            seen = streak[k] >= FOR_MIN if mode == "auto" else (t % 15 == phase and BREACH[k](m))       # alert rule needs 3 minutes; a human looks every 15 minutes
            if not seen or k in busy or k in res: continue
            if not S[k]: fa += 1; continue
            busy.add(k); det[k] = t; steps = RUNBOOK[playbook(k)["runbook"]]; machine = sum(x[1] for x in steps)
            if mode == "auto": total = 1 + machine                                                      # 1 minute from alert to action (stated assumption)
            else: total = int(rng.integers(1, 6)) + int(rng.integers(5, 13)) + sum(int(rng.integers(2, 7)) + x[1] for x in steps)
            esc += int(total > playbook(k)["escalate_after_min"]); pend.append((t + total, k))
            if hook:
                tk[k] = call(hook, "POST", "/alerts", dict(alerts=[dict(labels=dict(alertname=k), startsAt=t)]))["id"]; at = t + 1
                for name, mm in steps: call(hook, "PATCH", f"/tickets/{tk[k]}", dict(comment=f"min {at}: ran '{name}' ({mm} min)")); at += mm
    return dict(det=det, res=res, fa=fa, esc=esc)

def part_e():
    assert playbook("gpu_xid_error")["runbook"] is None and playbook("feed")["runbook"] == "feed"          # unknown alert goes to a human, known alert to its runbook
    srv = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Hook); SERVERS.append(srv); threading.Thread(target=srv.serve_forever, daemon=True).start()
    base = f"http://127.0.0.1:{srv.server_address[1]}"
    runs = {"auto": [simulate("auto", s, base if s == 0 else None) for s in range(30)], "manual": [simulate("manual", s) for s in range(30)]}
    for mode, rs in runs.items():
        assert all(set(r["res"]) == set(INCS) and r["fa"] == 0 for r in rs), mode                       # the 4 faults are all detected and fixed, no alert on a healthy minute
        assert all(r["det"][k] >= INCS[k] for r in rs for k in INCS)
    rows = []
    for k, st in INCS.items():
        row = dict(incident=k, start_min=st)
        for mode, rs in runs.items():
            d, r = np.array([x["det"][k] - st for x in rs]), np.array([x["res"][k] - st for x in rs])
            row.update({f"{mode}_MTTD": round(d.mean(), 1), f"{mode}_MTTR": round(r.mean(), 1), f"{mode}_MTTR_p90": float(np.percentile(r, 90))})
        rows.append(row)
    tot = {m: (np.mean([x["det"][k] - INCS[k] for x in rs for k in INCS]), np.mean([x["res"][k] - INCS[k] for x in rs for k in INCS])) for m, rs in runs.items()}
    rows.append(dict(incident="all four", start_min="-", **{f"{m}_{n}": round(tot[m][i], 1) for m in runs for i, n in enumerate(["MTTD", "MTTR"])}, auto_MTTR_p90="-", manual_MTTR_p90="-"))
    print("E1. minutes from fault start, mean over 30 simulated days (seeds). MTTD = to alert or first human look, MTTR = to a passing health check.")
    show(pd.DataFrame(rows), "E_mttd_mttr")
    print(f"E2. automated MTTR is {tot['manual'][1] / tot['auto'][1]:.1f}x shorter than manual; automated MTTD is {tot['manual'][0] / max(tot['auto'][0], 1e-9):.1f}x shorter. Escalations (restore time above the playbook limit): auto {sum(r['esc'] for r in runs['auto'])}, manual {sum(r['esc'] for r in runs['manual'])} of {30 * 4} incidents each.")
    print("playbook (alert -> runbook, owner, severity, escalate after min):"); show(pd.DataFrame([dict(alert=a, **playbook(a)) for a in list(PLAYBOOK) + ["gpu_xid_error"]]), "E_playbook")
    print("E3. toolchain: alert -> ticket JSON -> action -> closed ticket (seed 0, local webhook). One full ticket:"); print(json.dumps(TICKETS["OPS-2"], indent=1)[:1200])
    assert len(TICKETS) == 4 and all(t["status"] == "closed" and len(t["timeline"]) >= 4 for t in TICKETS.values())
    print("tickets:", {t["id"]: (t["title"], t["status"], t["opened_min"], t["closed_min"]) for t in TICKETS.values()})
    json.dump(TICKETS, open(f"{OUT}/E_tickets.json", "w"), indent=1)

part("E runbooks and playbooks", part_e)

## Part F. Compute parameters and GPU sharing like Run:AI

Question: how do batch size, sequence length, and precision change speed and cost? How does a scheduler raise utilization? First, a CPU measurement of the three compute parameters. The shapes match a GPU. The numbers do not. Then a scheduler simulation: 8 GPUs, two teams, a guaranteed quota of 4 GPUs each.

Source: `maximizing-gpu-utilization-with-nvidia-runai-and-nvidia-nim.txt`. It says that Run:AI gives inference the highest priority, so training never preempts it. Training can checkpoint and resume. GPU fractions with bin packing fill partly used GPUs first. Dynamic fractions use a request and a limit. The source does not describe quota, over-quota, or preemption. Those rules are this notebook's model. The simulation shows the idea. It is not Run:AI code.

In [ ]:
# 10. Part F (1): compute parameters on the CPU. Shapes (batch, sequence length, precision, cost) are the lesson. These are not T4 numbers.
def best_of(fn, n=5):
    ts = []
    for _ in range(n): t = time.perf_counter(); fn(); ts.append(time.perf_counter() - t)
    return min(ts)

def part_f1():
    rng = np.random.default_rng(SEED); Wm = rng.normal(size=(512, 2048)).astype(np.float32); PRICE = 1.0   # cost units per device-hour: an assumption
    rows = []
    for b in [1, 4, 16, 64, 256]:
        x = rng.normal(size=(b, 512)).astype(np.float32); t = best_of(lambda: x @ Wm, 30)
        rows.append(dict(batch=b, ms=round(t * 1e3, 3), rows_per_s=int(b / t), cost_per_1M_busy=round(PRICE / 3600 / (b / t) * 1e6, 4), cost_per_1M_at_30pct_util=round(PRICE / 3600 / (b / t) * 1e6 / 0.3, 4)))
    print("F1. batch size: one matrix product per batch. Cost per million rows at price 1.0 per device-hour, device busy 100% or 30% of the time.")
    show(pd.DataFrame(rows), "F_batch_size")
    print(f"rows per second at batch 256 are {rows[-1]['rows_per_s'] / rows[0]['rows_per_s']:.1f}x those at batch 1; at 30% utilization the cost per million rows is {rows[2]['cost_per_1M_at_30pct_util'] / rows[2]['cost_per_1M_busy']:.2f}x the busy cost.")
    rows = []
    for L in [128, 256, 512, 1024]:
        q = rng.normal(size=(8, L, 64)).astype(np.float32); t = best_of(lambda: q @ q.transpose(0, 2, 1), 5)
        rows.append(dict(max_seq_len=L, ms=round(t * 1e3, 2), score_matrix_MiB=round(8 * L * L * 4 / 2 ** 20, 2)))
    for i in range(1, len(rows)): rows[i]["time_vs_previous"] = round(rows[i]["ms"] / rows[i - 1]["ms"], 2)
    print("F2. maximum sequence length: the attention score matrix (batch 8, one head) grows with the square of the length. Memory is exact, time is measured.")
    show(pd.DataFrame(rows), "F_sequence_length"); assert all(rows[i]["score_matrix_MiB"] == 4 * rows[i - 1]["score_matrix_MiB"] for i in range(1, 4))
    x32 = rng.normal(size=(64, 512)).astype(np.float32); W16, x16 = Wm.astype(np.float16), x32.astype(np.float16)
    e = np.abs((x32 @ Wm) - (x16 @ W16).astype(np.float32)).max() / np.abs(x32 @ Wm).max()
    prec = [dict(precision="float32", weight_MiB=round(Wm.nbytes / 2 ** 20, 2), ms=round(best_of(lambda: x32 @ Wm) * 1e3, 3), max_rel_error=0.0),
            dict(precision="float16", weight_MiB=round(W16.nbytes / 2 ** 20, 2), ms=round(best_of(lambda: x16 @ W16) * 1e3, 3), max_rel_error=round(float(e), 5))]
    print("F3. precision settings: float16 halves memory and adds rounding error. The CPU has no fast float16 path, so its time is not a GPU time.")
    show(pd.DataFrame(prec), "F_precision"); assert prec[1]["weight_MiB"] * 2 == prec[0]["weight_MiB"]

part("F1 compute parameters", part_f1)

In [ ]:
# 11. Part F (2): a scheduler simulation. 8 GPUs, teams A and B, guaranteed quota 4 each. Our model of the idea, not Run:AI code.
G, QUOTA = 8, {"A": 4.0, "B": 4.0}
def make_jobs(seed=3):
    rng, jobs = np.random.default_rng(seed), []
    for team, n, lo, hi in [("A", 18, 0, 120), ("B", 6, 0, 120), ("A", 4, 150, 250), ("B", 16, 150, 250)]:
        for _ in range(n): jobs.append(dict(team=team, arr=int(rng.integers(lo, hi)), g=float(rng.choice([1, 1, 2, 4])), dur=int(rng.integers(10, 30)), infer=False))
    for team in "AB":
        for _ in range(2): jobs.append(dict(team=team, arr=int(rng.integers(0, 60)), g=float(rng.choice([0.25, 0.5])), dur=int(rng.integers(150, 250)), infer=True))   # small, long, never preempted
    for i, j in enumerate(sorted(jobs, key=lambda j: j["arr"])): j["id"] = i
    return sorted(jobs, key=lambda j: j["arr"])

def schedule(jobs, overquota=False, preempt=False, frac=False):
    free, run, pend, done, t = [1.0] * G, [], [], 0, 0
    J = [dict(j, rem=j["dur"], start=None, ls=0, alloc=[]) for j in jobs]; stats_ = dict(useful=0.0, alloc=0.0, viol=0, pre=0, lost=0.0, work=0.0)
    need = lambda j: j["g"] if (frac or j["g"] >= 1) else 1.0
    usage = lambda team: sum(sum(s for _, s in j["alloc"]) for j in run if j["team"] == team)
    def place(j):
        n = need(j)
        if n >= 1:
            gs = [i for i in range(G) if free[i] > 1 - 1e-9][: int(n)]
            if len(gs) < int(n): return False
            for i in gs: free[i] = 0.0
            j["alloc"] = [(i, 1.0) for i in gs]
        else:
            c = [i for i in range(G) if free[i] >= n - 1e-9]
            if not c: return False
            i = min(c, key=lambda i: (free[i], i)); free[i] -= n; j["alloc"] = [(i, n)]           # bin packing: fill the most used GPU first
        return True
    def release(j):
        for i, s in j["alloc"]: free[i] = min(1.0, free[i] + s)
        j["alloc"] = []
    while t < 3000 and (done < len(J)):
        pend += [j for j in J if j["arr"] == t]
        for j in sorted(pend, key=lambda j: (not j["infer"], not (usage(j["team"]) + need(j) <= QUOTA[j["team"]] + 1e-9), j["arr"], j["id"])):
            inq = usage(j["team"]) + need(j) <= QUOTA[j["team"]] + 1e-9
            if not (inq or overquota): continue
            if not place(j) and preempt and (inq or j["infer"]):
                for v in sorted([v for v in run if v["team"] != j["team"] and not v["infer"] and usage(v["team"]) > QUOTA[v["team"]] + 1e-9], key=lambda v: -v["ls"]):
                    release(v); run.remove(v); v["rem"] += 2; pend.append(v); stats_["pre"] += 1; stats_["lost"] += 2 * v["g"]      # checkpoint and resume: 2 minutes of work lost
                    if place(j): break
                else: continue
            if j["alloc"]:
                if j["start"] is None: j["start"] = t
                j["ls"] = t; pend.remove(j); run.append(j)
        for j in pend:                                                      # a job that fits the team quota but waits: the guarantee is broken
            if usage(j["team"]) + need(j) <= QUOTA[j["team"]] + 1e-9: stats_["viol"] += 1
        stats_["useful"] += sum(j["g"] for j in run) / G; stats_["alloc"] += sum(s for j in run for _, s in j["alloc"]) / G; stats_["work"] += sum(j["g"] for j in run)
        for j in list(run):
            j["rem"] -= 1
            if j["rem"] <= 0: release(j); run.remove(j); done += 1; j["end"] = t + 1
        t += 1
    return J, t, stats_

def part_f2():
    jobs = make_jobs(); total_work = sum(j["g"] * j["dur"] for j in jobs); rows, raw = [], []
    pols = [("static partition", dict()), ("over-quota, no preemption", dict(overquota=True)), ("fair share (over-quota + preemption)", dict(overquota=True, preempt=True)),
            ("fair share + GPU fractions", dict(overquota=True, preempt=True, frac=True))]
    for name, kw in pols:
        J, mk, s = schedule(jobs, **kw); assert all(j.get("end") for j in J)
        assert abs(s["work"] - (total_work + s["lost"])) < 1e-6, (s["work"], total_work, s["lost"])   # every GPU-minute that ran is accounted for
        tr = [j for j in J if not j["infer"]]; w = np.array([j["start"] - j["arr"] for j in tr]); wi = np.array([j["start"] - j["arr"] for j in J if j["infer"]])
        rows.append(dict(policy=name, makespan_min=mk, useful_pct=round(100 * s["useful"] / mk, 1), alloc_pct=round(100 * s["alloc"] / mk, 1), wait_mean=round(w.mean(), 1), wait_p95=round(np.percentile(w, 95), 1),
                         wait_infer=round(wi.mean(), 1), quota_viol_min=s["viol"], preemptions=s["pre"], cost_per_useful_h=round(G * mk / 60 / (total_work / 60), 2)))
        raw += [dict(policy=name, **{k: j[k] for k in ("id", "team", "arr", "g", "dur", "infer", "start", "end")}) for j in J]
    print(f"F4. {len(jobs)} jobs over 8 GPUs. useful = GPU share the jobs ask for. alloc = GPU share taken (a 0.25 job on a whole GPU takes 1.0).")
    print("quota_viol_min = job-minutes a job that fits its team's quota waited. cost_per_useful_h = GPU-hours paid per useful GPU-hour (price 1.0).")
    show(pd.DataFrame(rows), "F_scheduler"); pd.DataFrame(raw).to_csv(f"{OUT}/F_scheduler_jobs.csv", index=False)
    r = {x["policy"]: x for x in rows}; best = min(rows, key=lambda x: x["wait_mean"])
    print(f"lowest mean training wait: {best['policy']} ({best['wait_mean']} min); quota guarantee broken (violation minutes above 0): {[x['policy'] for x in rows if x['quota_viol_min'] > 0]}")
    assert r["static partition"]["quota_viol_min"] == 0

part("F2 GPU scheduler", part_f2)

## Part G. NVIDIA AI Enterprise

Question: what is it for, which problems does it address, and where does it run? Sources: `ai-enterprise.txt` and `air-gap.txt`. The numbers in the table are NVIDIA claims. This notebook does not measure them.

In [ ]:
# 12. Part G: NVIDIA AI Enterprise, from the source files, and a picker that follows the table.
AIE = [("purpose", "reliable and secure path for AI in production: microservices, frameworks and libraries, GPU orchestration and infrastructure management, in a supported commercial suite", "ai-enterprise.txt"),
       ("problem: reliability", "extended-lifetime production branches and enterprise support", "ai-enterprise.txt"),
       ("problem: security and compliance", "secure software supply chain, vulnerability mitigation, software that meets global security and compliance standards", "ai-enterprise.txt"),
       ("problem: time to value", "NIM microservices and Blueprints, validated deployment guides; NVIDIA claims up to 10x GPU availability for data scientists", "ai-enterprise.txt"),
       ("problem: scale and cost", "NVIDIA claims up to 5x GPU utilization and up to 20x throughput on existing infrastructure; central policy, governance, visibility", "ai-enterprise.txt"),
       ("environment: data center and edge", "NVIDIA-Certified data center and edge systems", "ai-enterprise.txt"),
       ("environment: cloud", "marketplace offers on AWS (AMI), Microsoft Azure and Google Cloud (VMI); NIM on OCI with OKE or bare metal", "ai-enterprise.txt"),
       ("environment: no internet", "NIM can run air-gapped: NGC through a proxy, or a mirrored local model registry", "air-gap.txt"),
       ("starting options", "free hosted NIM APIs, free NGC downloads, 90-day trial license. The trial does not include Run:AI", "ai-enterprise.txt")]
def pick_env(internet=True, data_must_stay_onprem=False, edge=False, cloud=None):
    if not internet: return "air-gapped data center: NIM from a mirrored registry or an NGC proxy"
    if edge: return "NVIDIA-Certified edge system"
    if data_must_stay_onprem: return "NVIDIA-Certified data center system, on premises"
    if cloud in ("aws", "azure", "gcp"): return f"{cloud} marketplace offer (AI Enterprise image)"
    if cloud == "oci": return "OCI: NIM on OKE or bare metal"
    return "start with free hosted NIM APIs and NGC downloads, then the 90-day trial license"

import textwrap
def part_g():
    pd.DataFrame(AIE, columns=["aspect", "what the source says", "source file"]).to_csv(f"{OUT}/G_ai_enterprise.csv", index=False)
    for a, b, c in AIE: print(f"- {a} [{c}]"); print(textwrap.fill(b, 150, initial_indent="    ", subsequent_indent="    "))
    cases = [dict(internet=False), dict(edge=True), dict(data_must_stay_onprem=True), dict(cloud="azure"), dict(cloud="oci"), dict()]
    print("G2. environment picker:"); [print(f"  {c} -> {pick_env(**c)}") for c in cases]
    assert pick_env(internet=False, cloud="aws").startswith("air-gapped") and pick_env(edge=True, cloud="aws").endswith("edge system") and pick_env(cloud="gcp").startswith("gcp")
    assert pick_env(data_must_stay_onprem=True, cloud="aws").endswith("on premises") and pick_env().startswith("start with")

part("G AI Enterprise", part_g)

In [ ]:
# 13. Status table of all parts, then stop every server and print GPU memory in use.
print(pd.DataFrame(STATUS).to_string(index=False))
for s in SERVERS:
    try: s.shutdown(); s.server_close()
    except Exception as e: print("server stop failed:", type(e).__name__)
print("servers stopped:", len(SERVERS))
try: print("GPU memory in use (MiB):", subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=20, check=True).stdout.split())
except Exception: print("GPU memory in use: no GPU in this session (this notebook does not need one)")
import textwrap
print(f"total run time {time.time() - T0:.0f} s. Saved files:"); print(textwrap.fill(", ".join(sorted(f for f in os.listdir(OUT) if f.endswith((".csv", ".png", ".json", ".jsonl", ".txt")))), 150))

## What to look for

- **Part A.** The trace tree shows the path of one request. The slow span and the failed span come from span times and span status, not from the logs. The log line joins to the trace by `trace_id`. In the anomaly table, read precision, recall, and delay together. Each rate has a Wilson 95% interval. With 3 incidents in the test half, recall has a wide interval. Compare the EWMA row with the threshold row on false alarms and on delay.
- **Part B.** Read the delay table by row. A detector with `- 0/5` in a row is blind to that drift. Compare `acc_change` with the alarms: an alarm without an accuracy drop is a harmless drift, and an accuracy drop without an alarm from the input detectors is concept drift. The false-alarm table shows what the strict threshold costs in delay. The label detector has the 7 day lag built in.
- **Part C.** Compare triggers by accuracy and by `bad_retrains` together. A retrain only sees labels that are 7 days old, so after a sudden change the first retrain can use mostly old data. In `label_fault` the labels are wrong for 9 days: check which triggers retrain on them and what that does to `bad_retrains`. The McNemar column counts rows where one policy is right and `never` is wrong. In the release table, the offline test and the shadow gate cannot see the skew bug. Only the canary compares served answers with labels.
- **Part D.** Every check prints its own proof: three equal hashes, one changed row found by its row hash, the versions found for a deletion request, and the first bad entry of each forged log. A cut tail passes the chain. Only the anchor stored elsewhere catches it.
- **Part E.** Compare MTTD and MTTR per incident. MTTR for drift stays long in both modes, because the retrain takes 8 machine minutes. The human numbers are assumptions: change them and the gap changes. Check that the ticket has a timeline and a close minute.
- **Part F.** The CPU numbers show shapes only. Utilization needs both columns: useful (what jobs ask for) and allocated (what they take). The violation column is the guarantee: above 0 means a job that fits its team's quota waited. The scheduler is a model with stated rules. The source does not describe quota or preemption.
- **Part G.** The percentages in the table are NVIDIA claims. This notebook measures none of them.

Limits: one simulated service, one linear model, 5 seeds for drift and 3 for triggers, one scheduler workload with one seed. Human times are assumed. No GPU or API is used, so no part shows real serving latency.

## Exam rules this proves

1. **sizing.2, sizing.6**: Observability is logs, metrics, and traces joined by a trace ID. A trace shows which step is slow or failed. A dashboard and an alert rule with a persistence time (here 3 minutes) turn the signals into SLA breach detection. An EWMA reacts to a slow ramp. A threshold on each request raises more false alarms.
2. **sizing.2, sizing.6**: Drift detection needs the right detector. Input detectors (PSI, KS, prediction share) cannot see concept drift. Delayed-label accuracy sees it, but late. A drift alarm does not mean the accuracy fell.
3. **sizing.1**: Pick the retraining trigger by accuracy and by bad retrains. A retrain on labeled data that is 7 days old can make a model worse. A model registry with versions and aliases keeps the champion and the challenger apart.
4. **sizing.1**: A release has stages. Shadow and offline tests check that the model runs and looks sane. Only a canary with labels shows a train and serve skew. An automatic rollback limits how many wrong answers reach users.
5. **data.2**: Lineage walks from a prediction to a model version, run, data hash, and source rows. Provenance is the hash that proves the rows did not change. A deletion request needs that map. A hash-chained log gives auditability. Store its head hash outside the log. A model card and a risk tier record the model risk management decision.
6. **sizing.4, sizing.2**: A runbook is executable steps with a check. A playbook picks the runbook, the owner, and the escalation. Automation cuts MTTD and MTTR. The webhook, ticket, action, and close loop is toolchain integration.
7. **sizing.5**: Larger batches raise throughput. Longer sequences grow the attention matrix with the square of the length. Lower precision cuts memory and adds error. Cost per request is the price divided by throughput and by utilization. Run:AI raises utilization with inference priority, GPU fractions with bin packing, and checkpoint and resume (source: `maximizing-gpu-utilization-with-nvidia-runai-and-nvidia-nim.txt`).
8. **enterprise.2**: NVIDIA AI Enterprise is a supported production suite. It addresses reliability, security, time to value, and scale and cost. It runs on certified data center and edge systems, on cloud marketplaces, and air-gapped (source: `ai-enterprise.txt`, `air-gap.txt`).

### Exam terms

| Term | Plain line | Shown in part |
|---|---|---|
| Model lifecycle management | Training, registering, releasing, watching, and retiring versions. | C, D |
| lifecycle and release management | Stages with gates: offline test, shadow, canary, promote, rollback. | C |
| retraining triggers | Rules that start a retrain: never, weekly, drift alert, performance alert. | C |
| versioning | Each model has a version, a run, a data hash, and an alias. | C, D |
| AI monitoring concepts and metrics | Latency, error rate, drift scores, delayed-label accuracy. | A, B |
| observability | Seeing the inside of the service from logs, metrics, and traces. | A |
| post-training drift | The live data moves away from the training data. | B |
| latency | Time per request. Measured per span and per request. | A, F |
| throughput | Rows per second. Measured against batch size. | F |
| health indicators | Checks that say the service is fine: p95, error rate, PSI, rows per minute. | A, E |
| SLA breach detection | An alert when p95 stays over the limit for 3 minutes. | A, E |
| MTTD | Minutes from fault start to detection. | E |
| MTTR | Minutes from fault start to a passing health check. | E |
| AI Factory operations | Running AI as a continuous service: alerts, runbooks, scheduling (`ai-factories.txt`). | E, F |
| toolchain integration | Alert to ticket to action to closed ticket over HTTP and JSON. | E |
| runbooks | Executable steps for one incident, each with a check (`runbooks.txt`). | E |
| playbooks | The guide that maps an alert to a runbook, an owner, and an escalation. | E |
| Compute parameters that affect workload performance | Batch size, sequence length, and precision change speed, memory, and cost. | F |
| batch size | Rows per call. Raises throughput until the device is busy. | F |
| maximum sequence length | Longest input. Attention memory grows with its square. | F |
| precision settings | float32 or float16: memory against rounding error. | F |
| hardware utilization and cost tradeoffs | Cost per request is the price over throughput and utilization. | F |
| Run:AI compute management | Inference priority, GPU fractions with bin packing, checkpoint and resume (source file named in part F). | F |
| Observability components | Traces, metrics, logs, dashboards, alerts, drift and anomaly detectors. | A, B |
| dashboards | Four panels of the signals over time. | A |
| prompt/agent/retrieval tracing | Spans for prompt, retrieval, model call, and tool under one trace. | A |
| logging and alerting | JSON logs with a trace ID. Alert rules with a persistence time. | A, E |
| telemetry | The stream of logs, metrics, and traces (`telemetry-observability.txt`). | A |
| drift detection | PSI, KS, prediction share, delayed-label accuracy. | B |
| anomaly detection | Threshold and EWMA on request latency, scored with precision, recall, delay. | A |
| Data governance standards | Written rules that code checks, such as hash every row and log every release. | D |
| lineage | The chain from a prediction back to model, run, data, and rows. | D |
| provenance | The hash that proves where the rows came from and that they did not change. | D |
| auditability | A hash-chained, append-only log that shows forged entries. | D |
| model risk management | Model card, risk tier, controls per tier, deletion requests. | D |
| NVIDIA AI Enterprise | A supported commercial suite for AI in production. | G |
| enterprise problems addressed | Reliability, security, time to value, scale and cost. | G |
| deployment environments | Certified data center and edge, cloud marketplaces, air-gapped. | G |
| its purpose | The reliable and secure path for AI in production. | G |